# Lab 09 - Notebook 01: Bayes' Rule, Bag of Words and Naive Bayes

**In-class demonstration**

This notebook introduces the practical text-classification pipeline used with Multinomial Naive Bayes:

**raw text -> tokens -> numerical features -> Naive Bayes classifier -> prediction**

We will also compare **CountVectorizer** and **TfidfVectorizer**.

## 1. Bayes' Rule with a simple spam example

Suppose:

- $P(Spam)=0.20$
- $P(FREE\mid Spam)=0.75$
- $P(FREE\mid NotSpam)=0.0625$

Bayes' Rule updates our belief after observing the word **FREE**:

$$P(Spam\mid FREE)=
rac{P(FREE\mid Spam)P(Spam)}{P(FREE)}$$

where

$$P(FREE)=P(FREE\mid Spam)P(Spam)+P(FREE\mid NotSpam)P(NotSpam).$$

In [1]:
p_spam = 0.20
p_not_spam = 1 - p_spam
p_free_given_spam = 0.75
p_free_given_not_spam = 0.0625

p_free = (p_free_given_spam * p_spam) + (p_free_given_not_spam * p_not_spam)
p_spam_given_free = (p_free_given_spam * p_spam) / p_free

print(f"P(FREE) = {p_free:.3f}")
print(f"P(Spam | FREE) = {p_spam_given_free:.3f}")

P(FREE) = 0.200
P(Spam | FREE) = 0.750


### Interpretation

The prior probability of spam is 0.20. After observing **FREE**, the posterior becomes 0.75 in this example. The evidence has changed our belief.

## 2. Tokenization

A classifier cannot directly use a sentence as a numerical input. First, the sentence is broken into **tokens**.

Example:

`WIN a FREE holiday now!`

may become:

`['win', 'free', 'holiday', 'now']`

`CountVectorizer` can perform this tokenization internally. We can inspect the tokenizer/analyzer it uses.

In [2]:
from sklearn.feature_extraction.text import CountVectorizer

example_text = "WIN a FREE holiday now!"
vectorizer_demo = CountVectorizer()
analyzer = vectorizer_demo.build_analyzer()

print('Original text:')
print(example_text)
print()
print('Tokens produced by CountVectorizer:')
print(analyzer(example_text))

Original text:
WIN a FREE holiday now!

Tokens produced by CountVectorizer:
['win', 'free', 'holiday', 'now']


## 3. Bag of Words with CountVectorizer

After tokenization, we need numbers. **Bag of Words (BoW)** builds a vocabulary and records how often each vocabulary word appears in each document.

The word order is mostly ignored; the model works with word occurrence/count information.

In [3]:
messages = [
    "free prize win now",
    "claim your free offer",
    "win cash prize today",
    "team meeting at noon",
    "please review the project report",
    "see you at the meeting today"
]
labels = ['spam', 'spam', 'spam', 'ham', 'ham', 'ham']

count_vectorizer = CountVectorizer()
X_count = count_vectorizer.fit_transform(messages)

print('Vocabulary:')
print(count_vectorizer.get_feature_names_out())
print()
print('Bag-of-Words matrix:')
print(X_count.toarray())

Vocabulary:
['at' 'cash' 'claim' 'free' 'meeting' 'noon' 'now' 'offer' 'please'
 'prize' 'project' 'report' 'review' 'see' 'team' 'the' 'today' 'win'
 'you' 'your']

Bag-of-Words matrix:
[[0 0 0 1 0 0 1 0 0 1 0 0 0 0 0 0 0 1 0 0]
 [0 0 1 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1]
 [0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 1 0 0]
 [1 0 0 0 1 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0 1 1 1 0 0 1 0 0 0 0]
 [1 0 0 0 1 0 0 0 0 0 0 0 0 1 0 1 1 0 1 0]]


### What happened?

- **fit**: learned the vocabulary from the training messages.
- **transform**: converted each message into a numerical vector.
- A value such as `2` means the corresponding word occurred twice in that message.

For new/test messages, use **`transform()` only** so that the training vocabulary remains unchanged.

## 4. Multinomial Naive Bayes

Multinomial Naive Bayes is suitable for non-negative text features such as word counts. It estimates class priors and feature probabilities, then combines them to classify a new message.

The model makes a simplifying **conditional independence** assumption between features given the class.

In [4]:
from sklearn.naive_bayes import MultinomialNB
import pandas as pd

nb_count = MultinomialNB()
nb_count.fit(X_count, labels)

test_messages = [
    "claim your free prize",
    "project meeting today",
    "free cash offer now",
    "please review the report"
]

X_test_count = count_vectorizer.transform(test_messages)
count_pred = nb_count.predict(X_test_count)
count_prob = nb_count.predict_proba(X_test_count)
spam_index = list(nb_count.classes_).index('spam')

pd.DataFrame({
    'message': test_messages,
    'prediction': count_pred,
    'P(spam)': count_prob[:, spam_index]
})

,message,prediction,P(spam)
0,claim your free prize,spam,0.980960
1,project meeting today,ham,0.179031
2,free cash offer now,spam,0.971709
3,please review the report,ham,0.056274


## 5. TF-IDF representation

`TfidfVectorizer` also tokenizes text, but instead of keeping only raw word counts, it gives words **weights** based on their frequency in a document and how common they are across the document collection.

A common word appearing in many documents usually receives less relative importance than a more distinctive word.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Use the same messages and labels so that only the representation changes.
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(messages)

print(tfidf_vectorizer.get_feature_names_out())
print(X_tfidf.toarray())

nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_tfidf, labels)

X_test_tfidf = tfidf_vectorizer.transform(test_messages)
tfidf_pred = nb_tfidf.predict(X_test_tfidf)
tfidf_prob = nb_tfidf.predict_proba(X_test_tfidf)
spam_index_tfidf = list(nb_tfidf.classes_).index('spam')

pd.DataFrame({
    'message': test_messages,
    'Count prediction': count_pred,
    'Count P(spam)': count_prob[:, spam_index],
    'TF-IDF prediction': tfidf_pred,
    'TF-IDF P(spam)': tfidf_prob[:, spam_index_tfidf]
})

['at' 'cash' 'claim' 'free' 'meeting' 'noon' 'now' 'offer' 'please'
 'prize' 'project' 'report' 'review' 'see' 'team' 'the' 'today' 'win'
 'you' 'your']
[[0.         0.         0.         0.4720786  0.         0.
  0.57569557 0.         0.         0.4720786  0.         0.
  0.         0.         0.         0.         0.         0.4720786
  0.         0.        ]
 [0.         0.         0.52182349 0.42790272 0.         0.
  0.         0.52182349 0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.52182349]
 [0.         0.57569557 0.         0.         0.         0.
  0.         0.         0.         0.4720786  0.         0.
  0.         0.         0.         0.         0.4720786  0.4720786
  0.         0.        ]
 [0.44836665 0.         0.         0.         0.44836665 0.54677906
  0.         0.         0.         0.         0.         0.
  0.         0.         0.54677906 0.         0.         0.
  0.         0.        ]
 [0. 

,message,Count prediction,Count P(spam),TF-IDF prediction,TF-IDF P(spam)
0,claim your free prize,spam,0.980960,spam,0.749618
1,project meeting today,ham,0.179031,ham,0.379582
2,free cash offer now,spam,0.971709,spam,0.734953
3,please review the report,ham,0.056274,ham,0.313228


## 6. Key points from Notebook 01

- **Tokenization** splits text into tokens.
- **Vectorization** converts those tokens into numerical features.
- **CountVectorizer** creates Bag-of-Words count features.
- **TfidfVectorizer** creates weighted text features.
- **Multinomial Naive Bayes** uses the numerical text features to estimate the most likely class.
- The same classifier can behave differently when the text representation changes.